# Projet Deep Learning : Reconnaissance d'expressions faciales

Le but est d'apprendre à un réseau de neurones à reconnaître l'émotion d'un visage (joie, colère, tristesse...).
On avance étape par étape : d'abord les données, puis un modèle simple, puis un CNN qu'on essaie ensuite d'améliorer.

## Partie 1 : Les données

### 1.1 Installation et imports

Sur Colab, TensorFlow est déjà installé. Il manque juste `kagglehub` pour télécharger le dataset depuis Kaggle.

In [ ]:
%pip install -q kagglehub

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
import kagglehub

### 1.2 Le dataset : FER-2013

On utilise FER-2013, récupéré sur Kaggle ([version en images](https://www.kaggle.com/datasets/msambare/fer2013)). Ce sont des visages déjà recadrés, en 48×48 pixels et en niveaux de gris, avec 7 émotions : angry, disgust, fear, happy, neutral, sad, surprise.

Le dataset est déjà séparé en `train/` et `test/`, avec un sous-dossier par émotion, donc le nom du dossier donne le label.

In [ ]:
# Téléchargement (la première fois ça prend quelques secondes, ensuite c'est en cache)
data_dir = kagglehub.dataset_download("msambare/fer2013")
train_dir = os.path.join(data_dir, "train")
test_dir = os.path.join(data_dir, "test")

print("Dataset téléchargé dans :", data_dir)
print("Contenu :", os.listdir(data_dir))

### 1.3 Premier coup d'œil

On compte combien d'images il y a dans chaque classe, pour le train et le test.

In [ ]:
classes = sorted(os.listdir(train_dir))
print("Classes :", classes)

# On compte les fichiers dans chaque dossier
comptes = pd.DataFrame({
    "train": [len(os.listdir(os.path.join(train_dir, c))) for c in classes],
    "test": [len(os.listdir(os.path.join(test_dir, c))) for c in classes],
}, index=classes)
comptes.loc["TOTAL"] = comptes.sum()
comptes

Et on ouvre une image au hasard pour vérifier sa taille et son format :

In [ ]:
dossier = os.path.join(train_dir, "happy")
fichier = os.listdir(dossier)[0]
img = Image.open(os.path.join(dossier, fichier))

print("Fichier :", fichier)
print("Taille :", img.size, "| Mode :", img.mode, "(L = niveaux de gris)")

plt.imshow(img, cmap="gray")
plt.title("happy")
plt.axis("off")
plt.show()

### 1.4 Toutes les images ont-elles le même format ?

On n'a regardé qu'une seule image, donc on vérifie sur tout le dataset.

In [ ]:
tailles = set()
modes = set()
extensions = set()

for dossier in [train_dir, test_dir]:
    for c in classes:
        for f in os.listdir(os.path.join(dossier, c)):
            img = Image.open(os.path.join(dossier, c, f))
            tailles.add(img.size)
            modes.add(img.mode)
            extensions.add(os.path.splitext(f)[1])

print("Tailles trouvées :", tailles)
print("Modes trouvés :", modes)
print("Extensions trouvées :", extensions)

### 1.5 Répartition des classes

On trace le nombre d'images par émotion. Si certaines classes ont beaucoup plus d'images que d'autres, le modèle risque de les favoriser.

In [ ]:
comptes_classes = comptes.drop("TOTAL")

comptes_classes.plot(kind="bar", figsize=(9, 4))
plt.title("Nombre d'images par classe")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=0)
plt.show()

# En pourcentage, c'est plus parlant
pourcentages = (comptes_classes["train"] / comptes_classes["train"].sum() * 100).round(1)
print("Part de chaque classe dans le train (%) :")
print(pourcentages)
print()
print("Ratio classe la plus fréquente / la moins fréquente :",
      round(comptes_classes["train"].max() / comptes_classes["train"].min(), 1))

Le dataset est très déséquilibré. `happy` représente environ 25 % du train, alors que `disgust` n'en fait que 1.5 %. Le test a la même répartition.

Un modèle qui répondrait toujours "happy" aurait donc déjà ~25 % d'accuracy. C'est pour ça qu'on regardera aussi la matrice de confusion et les scores par classe, et pas seulement l'accuracy.

### 1.6 À quoi ressemblent les visages ?

On affiche quelques images au hasard pour chaque émotion (une ligne par classe).

In [ ]:
nb_par_classe = 8
np.random.seed(0)

fig, axes = plt.subplots(len(classes), nb_par_classe, figsize=(nb_par_classe * 1.3, len(classes) * 1.5))

for i, c in enumerate(classes):
    fichiers = os.listdir(os.path.join(train_dir, c))
    choix = np.random.choice(fichiers, nb_par_classe, replace=False)
    for j, f in enumerate(choix):
        img = Image.open(os.path.join(train_dir, c, f))
        axes[i, j].imshow(img, cmap="gray")
        axes[i, j].axis("off")
    axes[i, 0].set_title(c, loc="left", fontsize=11)

plt.tight_layout()
plt.show()

Les visages sont à peu près centrés, mais l'angle, l'éclairage et l'âge varient beaucoup. Certaines images sont floues, en partie cachées (mains, lunettes) ou ont un filigrane.

Certaines émotions se ressemblent aussi pas mal (`fear` et `surprise`, `sad` et `neutral`, `angry` et `disgust`), le modèle risque de les confondre.

### 1.7 Préparation des données

Le réseau a besoin de tableaux de nombres, pas de fichiers JPG. Les étapes :
1. charger les images dans des tableaux numpy (`X` pour les images, `y` pour les labels)
2. redimensionner (pas besoin ici, tout est déjà en 48×48)
3. normaliser les pixels
4. encoder les labels
5. découper en train / validation / test

#### Chargement

Pour chaque image, on met ses pixels dans `X` et le numéro de sa classe dans `y` (0 = angry, 1 = disgust, ..., 6 = surprise). Le numéro correspond à la position du dossier dans la liste `classes`.

In [ ]:
X_train_full, y_train_full = [], []
X_test, y_test = [], []

for i, c in enumerate(classes):
    for f in os.listdir(os.path.join(train_dir, c)):
        X_train_full.append(np.array(Image.open(os.path.join(train_dir, c, f))))
        y_train_full.append(i)
    for f in os.listdir(os.path.join(test_dir, c)):
        X_test.append(np.array(Image.open(os.path.join(test_dir, c, f))))
        y_test.append(i)

X_train_full = np.array(X_train_full)
y_train_full = np.array(y_train_full)
X_test = np.array(X_test)
y_test = np.array(y_test)

print("X_train_full :", X_train_full.shape, X_train_full.dtype)
print("y_train_full :", y_train_full.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

On a 28 709 images de 48×48 pour le train. Keras attend aussi une dimension pour les canaux (1 en gris, 3 en couleur), donc on passe de `(N, 48, 48)` à `(N, 48, 48, 1)`.

In [ ]:
X_train_full = X_train_full.reshape(-1, 48, 48, 1)
X_test = X_test.reshape(-1, 48, 48, 1)

print("X_train_full :", X_train_full.shape)
print("X_test :", X_test.shape)

#### Normalisation

Les pixels vont de 0 (noir) à 255 (blanc). On les divise par 255 pour les ramener entre 0 et 1.

Si on laisse des valeurs jusqu'à 255, les sommes pondérées et les gradients deviennent très grands et l'entraînement est instable. Entre 0 et 1, c'est beaucoup plus stable.

In [ ]:
# On garde une copie des valeurs brutes juste pour comparer
pixels_avant = X_train_full[:1000].flatten()

X_train_full = X_train_full.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

pixels_apres = X_train_full[:1000].flatten()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(pixels_avant, bins=50)
axes[0].set_title("Pixels avant normalisation (0 → 255)")
axes[1].hist(pixels_apres, bins=50, color="tab:orange")
axes[1].set_title("Pixels après normalisation (0 → 1)")
plt.show()

print("Min / max après :", X_train_full.min(), "/", X_train_full.max())

L'histogramme garde la même forme, seule l'échelle change, donc on ne perd aucune information.

#### Découpage train / validation / test

Le dataset fournit déjà un jeu de test. On n'y touche pas avant l'évaluation finale, sinon le score final serait trop optimiste.

Pour suivre l'entraînement et comparer les modèles, on prend 15 % du train au hasard comme jeu de validation. Avec `stratify=y_train_full`, la répartition des classes reste la même dans les deux. Sans ça, vu qu'il n'y a que 436 images `disgust`, on pourrait en avoir très peu en validation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15,
    stratify=y_train_full,
    random_state=42,
)

print("Train      :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

On vérifie que les pourcentages sont presque identiques :

In [ ]:
repartition = pd.DataFrame({
    "train (%)": np.bincount(y_train) / len(y_train) * 100,
    "validation (%)": np.bincount(y_val) / len(y_val) * 100,
    "test (%)": np.bincount(y_test) / len(y_test) * 100,
}, index=classes).round(1)
repartition

#### Encodage des labels (one-hot)

Pour l'instant un label est un numéro, par exemple `3` pour happy. Le réseau va sortir 7 probabilités (une par classe), donc on transforme chaque label en vecteur de 7 cases avec un 1 à la bonne place :

`3` → `[0, 0, 0, 1, 0, 0, 0]`

Ça évite aussi que le réseau pense qu'il y a un ordre entre les classes. On garde les labels en numéros (`y_train`, `y_val`, `y_test`) pour la matrice de confusion plus tard.

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train_oh = to_categorical(y_train, num_classes=len(classes))
y_val_oh = to_categorical(y_val, num_classes=len(classes))
y_test_oh = to_categorical(y_test, num_classes=len(classes))

print("y_train :", y_train.shape, "→ y_train_oh :", y_train_oh.shape)

### 1.8 Vérification finale

On affiche quelques images comme le réseau va les recevoir (normalisées), avec leur label en numéro, en one-hot et en nom, pour vérifier que tout correspond.

In [ ]:
np.random.seed(1)
indices = np.random.choice(len(X_train), 6, replace=False)

fig, axes = plt.subplots(1, 6, figsize=(15, 3.5))
for ax, idx in zip(axes, indices):
    ax.imshow(X_train[idx].squeeze(), cmap="gray")
    ax.set_title(f"{classes[y_train[idx]]} ({y_train[idx]})\n{y_train_oh[idx].astype(int)}", fontsize=9)
    ax.axis("off")
plt.show()

Ce qu'on a à la fin :

| Jeu | Variable | Forme | Rôle |
|---|---|---|---|
| Train | `X_train`, `y_train_oh` | (24 402, 48, 48, 1) | apprendre les poids |
| Validation | `X_val`, `y_val_oh` | (4 307, 48, 48, 1) | suivre l'entraînement, comparer les modèles |
| Test | `X_test`, `y_test_oh` | (7 178, 48, 48, 1) | évaluation finale uniquement |

## Partie 2 : Modèle de référence (réseau dense)

Avant le CNN, on fait un réseau dense simple. Il ne sera pas très bon mais il sert de point de comparaison, le CNN devra faire mieux.

Architecture : Image → Flatten → Dense(128) + ReLU → Dense(7) + Softmax

### 2.1 Comment ça marche

**Flatten.** Une couche Dense prend un vecteur, donc on déroule l'image 48×48 en 2 304 valeurs. On perd au passage la position des pixels les uns par rapport aux autres.

**Poids et biais.** Chaque neurone calcule une somme pondérée de toutes ses entrées, `z = w₁x₁ + ... + wₙxₙ + b`. Ce sont les poids `w` et le biais `b` que le réseau apprend.

**Activation.** On applique ReLU, `max(0, z)`, après la somme. Sans fonction non linéaire, empiler des couches reviendrait à faire une seule opération linéaire.

**Propagation avant.** L'image traverse les couches une par une jusqu'à la sortie.

**Sortie.** La dernière couche a 7 neurones avec Softmax, qui transforme les sorties en 7 probabilités dont la somme fait 1. La classe prédite est celle avec la plus grande proba.

**Loss.** On utilise la categorical cross-entropy, `-log(p de la vraie classe)`. Plus la proba donnée à la bonne classe est faible, plus la loss est grande.

**Rétropropagation.** On calcule le gradient de la loss par rapport à chaque poids, en partant de la sortie et en remontant les couches (règle de dérivation en chaîne).

**Descente de gradient.** On modifie chaque poids dans le sens qui fait baisser la loss, `w ← w - lr × gradient`. On le fait par batchs d'images, sur plusieurs époques (une époque = un passage sur tout le train). On utilise Adam, qui adapte le pas pour chaque poids.

### 2.2 Construction du modèle

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# On fixe la graine aléatoire pour avoir des résultats reproductibles
keras.utils.set_random_seed(42)

baseline = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(7, activation="softmax"),
])

baseline.summary()

Pour le nombre de paramètres :
- `Flatten` : 0, il ne fait que réorganiser les pixels
- `Dense(128)` : 2 304 × 128 + 128 biais = 295 040
- `Dense(7)` : 128 × 7 + 7 = 903

Presque tout est dans la première couche, parce que chaque neurone est relié aux 2 304 pixels.

### 2.3 Entraînement

On entraîne 20 époques avec Adam et des batchs de 64, en suivant la loss et l'accuracy sur la validation.

In [ ]:
baseline.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_baseline = baseline.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=20,
    batch_size=64,
    verbose=2,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_baseline.history["loss"], label="train")
axes[0].plot(history_baseline.history["val_loss"], label="validation")
axes[0].set_title("Loss — modèle dense")
axes[0].set_xlabel("Époque")
axes[0].legend()

axes[1].plot(history_baseline.history["accuracy"], label="train")
axes[1].plot(history_baseline.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy — modèle dense")
axes[1].set_xlabel("Époque")
axes[1].legend()

plt.show()

print("Accuracy validation finale :", round(history_baseline.history["val_accuracy"][-1], 3))
print("Pour rappel, toujours répondre 'happy' donnerait ~0.25")

Le modèle dense arrive à ~37 % en validation. C'est mieux que les 25 % de "toujours happy", mais ça reste faible.

La validation n'avance plus après ~8 époques alors que le train continue de monter, c'est un début de surapprentissage. Et comme il voit l'image comme une simple liste de pixels, il ne peut pas repérer une bouche ou des sourcils s'ils ne sont pas toujours au même endroit.

C'est le score à battre avec le CNN.

## Partie 3 : Construction d'un CNN

### 3.1 Les briques d'un CNN

**Convolution.** Un filtre est une petite grille de poids (ici 3×3) qu'on fait glisser sur l'image. À chaque position on multiplie les pixels par les poids et on additionne, donc le résultat est grand quand la zone ressemble au motif du filtre. Les poids des filtres sont appris pendant l'entraînement.

**Feature maps.** Chaque filtre produit une nouvelle image qui montre où son motif apparaît. 32 filtres donnent 32 feature maps.

**Kernel.** C'est la taille du filtre. On prend 3×3, ça fait peu de paramètres et en empilant les couches le réseau voit quand même de grandes zones.

**Stride.** Le pas du filtre. On le laisse à 1 et c'est le pooling qui réduit la taille.

**Padding.** Avec `padding="same"`, on ajoute des zéros autour de l'image pour que la sortie garde la même taille.

**ReLU.** Comme avant, `max(0, z)` après chaque convolution.

**Pooling.** Le MaxPooling 2×2 garde le max de chaque carré de 2×2, donc divise la taille par 2. Ça réduit le calcul et rend le réseau moins sensible aux petits décalages.

**Flatten + Dense.** À la fin, on déroule les feature maps et des couches Dense décident de l'émotion, avec Dense(7) + Softmax en sortie.

### 3.2 Notre architecture

On empile 3 blocs Conv + Pool avec 32, 64 puis 128 filtres. Les premières couches repèrent des choses simples (bords, contrastes), les suivantes les combinent en formes plus complexes (yeux, bouche, sourcils).

On double le nombre de filtres à chaque fois que le pooling divise la taille par 2. Après 3 poolings on est à 6×6, on ne descend pas plus bas pour des images de 48×48.

In [ ]:
keras.utils.set_random_seed(42)

cnn = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),

    # Bloc 1 : motifs simples (bords, contrastes)
    layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    # Bloc 2 : motifs un peu plus complexes
    layers.Conv2D(64, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    # Bloc 3 : parties du visage
    layers.Conv2D(128, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    # Classification
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(7, activation="softmax"),
])

cnn.summary()

### 3.3 Évolution des dimensions et nombre de paramètres

Pour une Conv2D, le nombre de paramètres vaut `(kernel × kernel × canaux en entrée + 1) × nb filtres`, le +1 étant le biais de chaque filtre.

| Couche | Sortie | Calcul des paramètres | Paramètres |
|---|---|---|---|
| Entrée | 48 × 48 × 1 | | 0 |
| Conv2D 32 filtres 3×3 | 48 × 48 × 32 | (3×3×1 + 1) × 32 | 320 |
| MaxPooling 2×2 | 24 × 24 × 32 | | 0 |
| Conv2D 64 filtres 3×3 | 24 × 24 × 64 | (3×3×32 + 1) × 64 | 18 496 |
| MaxPooling 2×2 | 12 × 12 × 64 | | 0 |
| Conv2D 128 filtres 3×3 | 12 × 12 × 128 | (3×3×64 + 1) × 128 | 73 856 |
| MaxPooling 2×2 | 6 × 6 × 128 | | 0 |
| Flatten | 4 608 | | 0 |
| Dense 128 | 128 | 4 608 × 128 + 128 | 589 952 |
| Dense 7 (Softmax) | 7 | 128 × 7 + 7 | 903 |
| Total | | | 683 527 |

Avec `padding="same"`, la taille ne change pas dans les Conv2D, c'est le pooling qui la divise par 2.

Les convolutions ont peu de paramètres parce qu'un même filtre est utilisé sur toute l'image, alors que la première Dense en a 86 % à elle seule. On s'en souviendra si le modèle sur-apprend.

### 3.4 À quoi ressemblent les feature maps ?

On fait passer un visage dans la première convolution et on affiche les 32 feature maps. Le réseau n'est pas encore entraîné, donc les filtres sont aléatoires. On refera la même chose après l'entraînement pour comparer.

In [ ]:
# Un mini-modèle qui s'arrête après la première couche de convolution
extracteur = keras.Model(inputs=cnn.inputs, outputs=cnn.layers[0].output)

visage = X_train[0:1]  # on garde la dimension batch : (1, 48, 48, 1)
feature_maps = extracteur.predict(visage, verbose=0)
print("Forme des feature maps :", feature_maps.shape)

plt.imshow(visage[0].squeeze(), cmap="gray")
plt.title(f"Image d'entrée : {classes[y_train[0]]}")
plt.axis("off")
plt.show()

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, :, :, i], cmap="viridis")
    ax.axis("off")
plt.suptitle("Les 32 feature maps de la 1re couche (filtres aléatoires, avant entraînement)")
plt.show()

Chaque filtre fait ressortir quelque chose de différent, certains les contours, d'autres les zones claires ou sombres. Beaucoup de feature maps sont presque noires à cause du ReLU, qui met les valeurs négatives à 0. La taille reste bien 48×48 grâce au padding.

## Partie 4 : Entraînement du CNN

### 4.1 Nos choix

On garde les mêmes réglages que pour le modèle dense, pour que la comparaison soit juste :
- loss `categorical_crossentropy`, la loss habituelle avec Softmax et des labels one-hot
- Adam avec le learning rate par défaut (0.001), qui marche bien sans trop de réglages
- batchs de 64, soit ~382 mises à jour par époque
- 20 époques, assez pour voir le modèle converger et voir s'il sur-apprend
- l'accuracy comme métrique, en plus des scores par classe à cause du déséquilibre

### 4.2 Entraînement

Sur Colab, on peut activer le GPU pour aller plus vite.

In [ ]:
cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_cnn = cnn.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=20,
    batch_size=64,
    verbose=2,
)

### 4.3 Courbes d'apprentissage

On trace le CNN et le modèle dense sur les mêmes graphiques.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history_cnn.history["loss"], label="CNN - train", color="tab:blue")
axes[0].plot(history_cnn.history["val_loss"], label="CNN - validation", color="tab:orange")
axes[0].plot(history_baseline.history["val_loss"], label="Dense - validation", color="gray", linestyle="--")
axes[0].set_title("Loss")
axes[0].set_xlabel("Époque")
axes[0].legend()

axes[1].plot(history_cnn.history["accuracy"], label="CNN - train", color="tab:blue")
axes[1].plot(history_cnn.history["val_accuracy"], label="CNN - validation", color="tab:orange")
axes[1].plot(history_baseline.history["val_accuracy"], label="Dense - validation", color="gray", linestyle="--")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Époque")
axes[1].legend()

plt.show()

meilleure_epoque = int(np.argmin(history_cnn.history["val_loss"]))
print("Meilleure époque (val_loss la plus basse) :", meilleure_epoque + 1)
print("  val_accuracy à cette époque :", round(history_cnn.history["val_accuracy"][meilleure_epoque], 3))
print("val_accuracy à la fin :", round(history_cnn.history["val_accuracy"][-1], 3))
print("train_accuracy à la fin :", round(history_cnn.history["accuracy"][-1], 3))

Le CNN fait bien mieux que le modèle dense (~53 % contre ~37 % en validation), parce que les convolutions repèrent les yeux et la bouche où qu'ils soient dans l'image.

Par contre il sur-apprend vite. Après l'époque 5, la loss de validation remonte (de 1.25 à plus de 3) alors que l'accuracy de train monte à 93 % et que la validation reste vers 52 %. Il apprend le train par cœur, ce qui est possible avec ~680 000 paramètres pour 24 000 images.

La loss de validation qui explose veut aussi dire qu'il devient trop sûr de lui : quand il se trompe, c'est avec une proba proche de 1.

### 4.4 Les feature maps après entraînement

Même visage et même couche qu'en 3.4, mais avec les filtres appris.

In [ ]:
# Le mini-modèle partage les poids du CNN, donc il est entraîné lui aussi
feature_maps = extracteur.predict(visage, verbose=0)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, :, :, i], cmap="viridis")
    ax.axis("off")
plt.suptitle("Les 32 feature maps de la 1re couche (après entraînement)")
plt.show()

Les feature maps sont plus ciblées qu'avant. Plusieurs filtres réagissent surtout aux yeux, aux sourcils et à la bouche, donc aux zones qui portent l'expression. Le réseau a appris tout seul où regarder.

## Partie 5 : Évaluation et analyse des erreurs

On passe sur le jeu de test, que les modèles n'ont jamais vu. Les expériences de la partie 6 seront comparées sur la validation, et seul le modèle final repassera sur le test.

### 5.1 Score global sur le test

In [ ]:
loss_dense, acc_dense = baseline.evaluate(X_test, y_test_oh, verbose=0)
loss_cnn, acc_cnn = cnn.evaluate(X_test, y_test_oh, verbose=0)

pd.DataFrame({
    "loss test": [loss_dense, loss_cnn],
    "accuracy test": [acc_dense, acc_cnn],
}, index=["Modèle dense", "CNN"]).round(3)

Sur le test, le CNN fait ~52 % contre ~38 % pour le modèle dense. C'est proche des scores de validation, donc la validation donnait une bonne idée du résultat.

Par contre la loss du CNN (~3.2) est pire que celle du dense (~1.6). C'est le surapprentissage : il donne des probas extrêmes même quand il a tort.

### 5.2 Matrice de confusion

La matrice de confusion montre, pour chaque vraie classe (ligne), la classe prédite (colonne). La diagonale correspond aux bonnes réponses.

On la normalise par ligne pour avoir des pourcentages, sinon `happy`, qui a beaucoup plus d'images, prendrait toute la place.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

probas_test = cnn.predict(X_test, verbose=0)   # (7178, 7) : les 7 probas pour chaque image
y_pred = np.argmax(probas_test, axis=1)        # la classe avec la plus grande proba

cm = confusion_matrix(y_test, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
plt.title("Matrice de confusion du CNN (test, normalisée par ligne)")
plt.show()

`happy` (75 %) et `surprise` (66 %) sont les mieux reconnues, ce sont des expressions très marquées. `sad`, `fear`, `angry` et `neutral` tournent autour de 40 %.

Les plus grosses confusions sont entre `sad`, `fear` et `neutral`, des expressions plus subtiles. `disgust` est souvent pris pour `angry` (23 %) et `surprise` pour `fear` (14 %), ce qui se comprend vu qu'elles se ressemblent.

### 5.3 Scores par classe

- Precision : quand le modèle dit "happy", combien de fois il a raison
- Recall : parmi les vraies images "happy", combien il en retrouve
- F1-score : la moyenne harmonique des deux
- Support : le nombre d'images de la classe dans le test

In [ ]:
print(classification_report(y_test, y_pred, target_names=classes, digits=2))

On retrouve la même chose que dans la matrice : `happy` (0.74) et `surprise` (0.68) ont les meilleurs F1, `fear` (0.37) et `sad` (0.39) les moins bons.

`neutral` a une precision plus haute que son recall, donc le modèle range beaucoup de visages neutres dans d'autres classes (sad, fear).

### 5.4 Exemples de prédictions

Sous chaque image, la vraie classe, la classe prédite et sa proba. En vert les bonnes réponses, en rouge les erreurs.

In [ ]:
np.random.seed(3)
indices = np.random.choice(len(X_test), 12, replace=False)

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
for ax, idx in zip(axes.flat, indices):
    vrai = classes[y_test[idx]]
    pred = classes[y_pred[idx]]
    proba = probas_test[idx, y_pred[idx]]
    ax.imshow(X_test[idx].squeeze(), cmap="gray")
    ax.set_title(f"vrai : {vrai}\npréd : {pred} ({proba:.2f})",
                 color="green" if vrai == pred else "red", fontsize=10)
    ax.axis("off")
plt.suptitle("Prédictions au hasard sur le test")
plt.show()

Les sourires et les bouches ouvertes sont bien reconnus. Le modèle donne souvent des probas de 1.00 même quand il a tort (par exemple l'enfant "angry" qui crie, prédit "happy"), encore à cause du surapprentissage.

### 5.5 Analyse des erreurs

On affiche seulement les erreurs, en commençant par celles où le modèle était le plus sûr de lui.

In [ ]:
erreurs = np.where(y_pred != y_test)[0]
print("Nombre d'erreurs :", len(erreurs), "sur", len(y_test))

# On trie les erreurs par probabilité décroissante
confiance = probas_test[erreurs, y_pred[erreurs]]
pires = erreurs[np.argsort(-confiance)][:12]

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
for ax, idx in zip(axes.flat, pires):
    ax.imshow(X_test[idx].squeeze(), cmap="gray")
    ax.set_title(f"vrai : {classes[y_test[idx]]}\npréd : {classes[y_pred[idx]]} ({probas_test[idx, y_pred[idx]]:.2f})",
                 color="red", fontsize=10)
    ax.axis("off")
plt.suptitle("Les 12 erreurs les plus confiantes")
plt.show()

Et les confusions les plus fréquentes (en nombre d'images) :

In [ ]:
cm_brute = confusion_matrix(y_test, y_pred)
np.fill_diagonal(cm_brute, 0)   # on enlève les bonnes réponses

confusions = []
for i in range(len(classes)):
    for j in range(len(classes)):
        confusions.append((classes[i], classes[j], cm_brute[i, j]))

pd.DataFrame(confusions, columns=["vraie classe", "prédite", "nb d'images"]) \
    .sort_values("nb d'images", ascending=False).head(8).reset_index(drop=True)

### 5.6 Bilan de l'analyse

Les confusions les plus fréquentes sont `sad → fear`, `neutral → sad`, `fear → sad` et `sad → neutral`. Le point faible du modèle, c'est surtout ces trois classes. Leurs expressions se ressemblent beaucoup et en 48×48, la différence tient parfois à quelques pixels.

Certaines erreurs viennent aussi du dataset : plusieurs images notées `surprise` ou `neutral` montrent un grand sourire, et il y a des dessins ou des visages de profil. Et le surapprentissage rend le modèle trop confiant.

## Partie 6 : Expérimentations et amélioration

Le principal problème de notre CNN est le surapprentissage. On fait quelques expériences pour l'améliorer, en changeant une seule chose à la fois pour savoir d'où vient la différence.

| Expérience | Ce qui change |
|---|---|
| 0 (référence) | le CNN de la partie 3 (kernel 3×3, 20 époques) |
| 1 | ajout de Dropout |
| 2 | kernel 3×3 → 5×5 |
| 3 | moins d'époques |

Toutes les expériences sont comparées sur la validation.

Une petite fonction pour tracer les courbes sans recopier le code à chaque fois :

In [ ]:
def tracer_courbes(history, titre):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="validation")
    axes[0].set_title("Loss — " + titre)
    axes[0].set_xlabel("Époque")
    axes[0].legend()
    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="validation")
    axes[1].set_title("Accuracy — " + titre)
    axes[1].set_xlabel("Époque")
    axes[1].legend()
    plt.show()

### 6.1 Expérience 1 : Dropout

Pendant l'entraînement, le Dropout éteint au hasard une partie des neurones à chaque batch. Le réseau ne peut plus compter sur des neurones précis pour mémoriser une image, donc il apprend des motifs plus généraux. Au moment de prédire, tous les neurones sont actifs.

On met 0.25 après chaque bloc de convolution, et 0.5 après la grosse couche Dense, là où il y a le plus de paramètres. Le reste ne change pas.

In [ ]:
keras.utils.set_random_seed(42)

cnn_dropout = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),

    layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Conv2D(64, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Conv2D(128, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7, activation="softmax"),
])

cnn_dropout.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history_dropout = cnn_dropout.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=20,
    batch_size=64,
    verbose=2,
)

In [ ]:
tracer_courbes(history_dropout, "Exp. 1 : Dropout")

Le surapprentissage a presque disparu. Les courbes de train et de validation restent proches et la loss de validation baisse jusqu'à la fin (1.14 à l'époque 18). L'accuracy de validation monte à ~57 %, contre ~53 % pour le CNN de base.

L'accuracy de train est beaucoup plus basse qu'avant (~59 % contre 93 %), ce qui est normal avec le Dropout. Les courbes montent encore à l'époque 20, donc le modèle n'a pas fini d'apprendre.

### 6.2 Expérience 2 : kernel 5×5

On repart du modèle avec Dropout et on passe les filtres de 3×3 à 5×5. Chaque filtre voit une zone plus grande du visage, mais le modèle a plus de paramètres (847 879 au lieu de 683 527) et il est plus lent. Avec le padding, les dimensions ne changent pas.

In [ ]:
keras.utils.set_random_seed(42)

cnn_k5 = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),

    layers.Conv2D(32, kernel_size=5, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Conv2D(64, kernel_size=5, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Conv2D(128, kernel_size=5, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7, activation="softmax"),
])

cnn_k5.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
print("Nombre de paramètres :", cnn_k5.count_params())

history_k5 = cnn_k5.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=20,
    batch_size=64,
    verbose=2,
)

In [ ]:
tracer_courbes(history_k5, "Exp. 2 : kernel 5×5")

Les deux kernels donnent la même accuracy (~57 % en validation), la différence est trop petite pour compter. Par contre le 5×5 a une loss de validation un peu moins bonne (1.21 contre 1.16) et recommence à sur-apprendre vers l'époque 13. Il est aussi 2 fois plus lent.

Sur des images de 48×48 les détails importants sont petits, et deux filtres 3×3 à la suite voient déjà une zone de 5×5. Un plus gros kernel n'aide pas ici.

### 6.3 Expérience 3 : moins d'époques

Une autre façon d'éviter le surapprentissage est de s'arrêter plus tôt. On reprend le CNN de base (sans Dropout), qui sur-apprenait à partir de l'époque 5, et on l'entraîne 5 époques au lieu de 20.

On ne le fait pas sur le modèle avec Dropout, qui apprenait encore à l'époque 20.

In [ ]:
keras.utils.set_random_seed(42)

cnn_court = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),

    layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    layers.Conv2D(64, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    layers.Conv2D(128, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(7, activation="softmax"),
])

cnn_court.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history_court = cnn_court.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=5,
    batch_size=64,
    verbose=2,
)

In [ ]:
tracer_courbes(history_court, "Exp. 3 : CNN de base, 5 époques")

On retrouve exactement le début de l'expérience 0 (même graine) et on s'arrête à 53.4 % en validation, avec une loss de 1.25. Le surapprentissage est évité, mais le modèle n'a pas eu le temps d'aller plus loin.

### 6.4 Tableau comparatif

Pour chaque expérience, on regarde le modèle à la fin de l'entraînement : la loss et l'accuracy de validation, et l'écart entre train et validation (plus il est grand, plus le modèle sur-apprend).

In [ ]:
experiences = {
    "0 - CNN de base (20 ép.)": history_cnn,
    "1 - + Dropout (20 ép.)": history_dropout,
    "2 - + kernel 5×5 (20 ép.)": history_k5,
    "3 - CNN de base (5 ép.)": history_court,
}

lignes = []
for nom, h in experiences.items():
    lignes.append({
        "expérience": nom,
        "val_loss": h.history["val_loss"][-1],
        "val_accuracy": h.history["val_accuracy"][-1],
        "train_accuracy": h.history["accuracy"][-1],
        "écart train - val": h.history["accuracy"][-1] - h.history["val_accuracy"][-1],
    })

tableau = pd.DataFrame(lignes).set_index("expérience").round(3)
tableau

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for nom, h in experiences.items():
    axes[0].plot(h.history["val_loss"], label=nom)
    axes[1].plot(h.history["val_accuracy"], label=nom)
axes[0].set_title("Loss de validation")
axes[1].set_title("Accuracy de validation")
for ax in axes:
    ax.set_xlabel("Époque")
    ax.legend()
plt.show()

### 6.5 Choix du modèle final

On garde l'expérience 1 (CNN 3×3 + Dropout, 20 époques). Elle fait la même accuracy que le 5×5, mais elle a la meilleure loss de validation, elle ne sur-apprend pas, et elle est plus légère et 2 fois plus rapide, ce qui compte pour la webcam en partie 9. S'arrêter plus tôt (exp. 3) évite aussi le surapprentissage, mais plafonne plus bas.

### 6.6 Le modèle final sur le test

On évalue le modèle retenu sur le test (une seule fois) et on le compare au modèle dense et au CNN de base.

In [42]:
cnn_final = cnn_dropout

loss_final, acc_final = cnn_final.evaluate(X_test, y_test_oh, verbose=0)

pd.DataFrame({
    "loss test": [loss_dense, loss_cnn, loss_final],
    "accuracy test": [acc_dense, acc_cnn, acc_final],
}, index=["Modèle dense", "CNN de base", "CNN final (Dropout)"]).round(3)

NameError: name 'cnn_dropout' is not defined

In [ ]:
from sklearn.metrics import f1_score

y_pred_final = np.argmax(cnn_final.predict(X_test, verbose=0), axis=1)

# F1-score par classe, avant / après
f1_avant = f1_score(y_test, y_pred, average=None)
f1_apres = f1_score(y_test, y_pred_final, average=None)

x = np.arange(len(classes))
plt.figure(figsize=(10, 4))
plt.bar(x - 0.2, f1_avant, width=0.4, label="CNN de base")
plt.bar(x + 0.2, f1_apres, width=0.4, label="CNN final")
plt.xticks(x, classes)
plt.ylabel("F1-score")
plt.title("F1-score par classe sur le test")
plt.legend()
plt.show()

cm_final = confusion_matrix(y_test, y_pred_final, normalize="true")
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm_final, display_labels=classes).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
plt.title("Matrice de confusion du CNN final (test)")
plt.show()

Le CNN final gagne 5.5 points sur le test (57.6 % contre 52.1 %) et sa loss est presque divisée par 3 (1.13 contre 3.19), donc ses probas sont beaucoup plus fiables. Le score est proche de celui de la validation.

Le F1 progresse pour angry, happy, neutral, sad et surprise, surtout pour `neutral` et `sad`. Il baisse pour `fear`, toujours confondue avec `sad`, et pour `disgust`, qui est maintenant souvent prédit `angry` (31 %). Avec seulement 436 images d'entraînement, le Dropout rend cette classe encore plus dure à apprendre.

Il n'est pas meilleur partout, mais il est meilleur globalement, donc c'est lui qu'on garde pour la suite.

On sauvegarde le modèle final pour le réutiliser dans les parties suivantes sans le ré-entraîner :

In [ ]:
cnn_final.save("modele_final.keras")
print("Modèle sauvegardé !")

## Partie 7 : Enrichissement avec la détection d'objets via YOLO

Jusqu'ici, notre CNN prend un visage déjà découpé. Sur une vraie photo, il faut d'abord trouver où sont les visages, c'est de la détection d'objets. On utilise YOLO pour ça.

### 7.1 Classification vs détection

La classification donne une classe pour toute l'image (notre CNN dit par exemple "happy, 0.92"). La détection donne, pour chaque objet de l'image, une boîte, une classe et un score.

### 7.2 Les notions clés

- **Bounding box** : le rectangle autour d'un objet, donné par ses coins `(x1, y1, x2, y2)`.
- **Confidence score** : à quel point le modèle est sûr qu'il y a un objet dans la boîte. On garde seulement les boîtes au-dessus d'un seuil (0.25 par défaut).
- **IoU** : l'aire de l'intersection de deux boîtes divisée par l'aire de leur union. Elle vaut 1 si les boîtes sont identiques, 0 si elles ne se touchent pas.
- **NMS** : le réseau propose souvent plusieurs boîtes pour le même objet. La NMS garde la meilleure et supprime celles qui la recouvrent trop (IoU au-dessus de 0.7 par défaut).
- **Precision / recall / mAP** : la precision est la part des boîtes détectées qui sont de vrais visages, le recall la part des vrais visages qui ont été trouvés. La mAP résume les deux sur tous les seuils. Il faut des images annotées pour la calculer, donc on ne le fait pas ici.

### 7.3 Le principe de YOLO

YOLO (You Only Look Once) fait passer l'image une seule fois dans un CNN. L'image est découpée en grille, et pour chaque case le réseau prédit des boîtes, leur score et leur classe. On enlève ensuite les boîtes peu confiantes, puis la NMS enlève les doublons. Tout se fait en un passage, donc c'est rapide et on peut l'utiliser en vidéo.

### 7.4 Quel modèle ?

Un YOLO classique est entraîné sur COCO, qui contient des personnes mais pas de classe visage. On prend donc un YOLOv8 nano déjà entraîné à détecter des visages ([arnabdhar/YOLOv8-Face-Detection](https://huggingface.co/arnabdhar/YOLOv8-Face-Detection)). Il tourne bien sur CPU et on l'utilise tel quel, sans l'entraîner.

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
from ultralytics.utils import ASSETS   # 2 photos d'exemple fournies avec ultralytics
from huggingface_hub import hf_hub_download
import matplotlib.patches as patches

chemin_yolo = hf_hub_download("arnabdhar/YOLOv8-Face-Detection", "model.pt")
yolo = YOLO(chemin_yolo)

print("Classes que ce YOLO sait détecter :", yolo.names)

### 7.5 Première détection

On teste sur une photo d'exemple fournie avec ultralytics. On écrit une petite fonction pour dessiner les boîtes, elle resservira plus loin :

In [ ]:
def dessiner_boites(image, boites, textes, titre=""):
    plt.figure(figsize=(10, 6))
    plt.imshow(image)
    for (x1, y1, x2, y2), texte in zip(boites, textes):
        plt.gca().add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="lime", linewidth=2))
        plt.text(x1, y1 - 5, texte, color="black", backgroundcolor="lime", fontsize=9)
    plt.title(titre)
    plt.axis("off")
    plt.show()


photo = Image.open(ASSETS / "zidane.jpg").convert("RGB")
resultat = yolo(photo, verbose=False)[0]

boites = resultat.boxes.xyxy.numpy()   # une ligne par visage : x1, y1, x2, y2
scores = resultat.boxes.conf.numpy()   # le score de confiance de chaque boîte

print("Nombre de visages détectés :", len(boites))
for b, s in zip(boites, scores):
    print("  boîte :", b.round(), "| confiance :", round(float(s), 2))

dessiner_boites(photo, boites, [f"visage {s:.2f}" for s in scores], "Détection de visages avec YOLO")

YOLO trouve les 2 visages. Celui de gauche a un score plus faible (~0.37) : il est de profil avec la bouche grande ouverte, donc il ressemble moins aux visages vus à l'entraînement.

### 7.6 La NMS en action

Pour voir l'effet de la NMS, on la désactive en mettant son seuil d'IoU à 1.0, et on baisse le seuil de confiance à 0.05 :

In [ ]:
sans_nms = yolo(photo, conf=0.05, iou=1.0, verbose=False)[0]
avec_nms = yolo(photo, conf=0.05, iou=0.7, verbose=False)[0]

print("Boîtes sans NMS :", len(sans_nms.boxes))
print("Boîtes avec NMS :", len(avec_nms.boxes))

dessiner_boites(photo, sans_nms.boxes.xyxy.numpy(), ["" for _ in sans_nms.boxes], "Sans NMS : plein de boîtes en double")

Sans NMS, on obtient une vingtaine de boîtes, presque toutes sur les 2 mêmes visages. La NMS garde seulement la meilleure de chaque groupe.

### 7.7 Calcul de l'IoU à la main

On calcule nous-mêmes l'IoU entre la meilleure boîte et les autres boîtes proposées :

In [ ]:
def iou(a, b):
    # coins du rectangle d'intersection
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    aire_a = (a[2] - a[0]) * (a[3] - a[1])
    aire_b = (b[2] - b[0]) * (b[3] - b[1])
    return intersection / (aire_a + aire_b - intersection)


boites_sans_nms = sans_nms.boxes.xyxy.numpy()
meilleure = boites_sans_nms[0]   # les boîtes sont triées par score décroissant

ious = [round(float(iou(meilleure, b)), 2) for b in boites_sans_nms[1:]]
print("IoU de chaque boîte avec la meilleure :", ious)

Les boîtes sur le même visage ont une IoU proche de 1 avec la meilleure, ce sont des doublons que la NMS supprime. Celles sur l'autre visage ont une IoU de 0, donc la NMS ne les compare pas à la meilleure et garde la meilleure d'entre elles.

### 7.8 Le seuil de confiance : precision vs recall

Avec un seuil bas, on rate moins de visages (meilleur recall) mais on risque de fausses détections. Avec un seuil haut, les détections sont plus sûres (meilleure precision) mais on rate les visages difficiles.

In [ ]:
for seuil in [0.25, 0.5]:
    r = yolo(photo, conf=seuil, verbose=False)[0]
    dessiner_boites(photo, r.boxes.xyxy.numpy(), [f"visage {s:.2f}" for s in r.boxes.conf.numpy()],
                    f"Seuil de confiance = {seuil} → {len(r.boxes)} visage(s)")

Avec un seuil à 0.5, le visage de profil (0.37) disparaît. On garde le seuil par défaut de 0.25 pour ne pas rater de visages.

## Partie 8 : Détection et analyse de plusieurs visages

On assemble tout :

Image → YOLO → boîtes → découpe de chaque visage → prétraitement → CNN → expression + proba

### 8.1 Le même prétraitement

Le CNN a été entraîné sur des images 48×48 en gris avec des pixels entre 0 et 1. Chaque visage découpé doit donc subir la même transformation, sinon le CNN reçoit quelque chose qu'il n'a jamais vu :
1. découper la boîte
2. passer en niveaux de gris
3. redimensionner en 48×48
4. diviser par 255
5. mettre en forme `(1, 48, 48, 1)`

On met aussi le seuil de la NMS à 0.5 au lieu de 0.7, parce qu'en vidéo YOLO laissait parfois une petite boîte dans une grande sur le même visage.

On recharge le modèle final sauvegardé en partie 6 :

In [ ]:
modele = keras.models.load_model("modele_final.keras")


def analyser_image(image):
    """Détecte les visages d'une image RGB et prédit l'expression de chacun."""
    # NMS à 0.5 au lieu de 0.7, sinon il reste parfois une petite boîte dans une grande
    resultat = yolo(image, iou=0.5, verbose=False)[0]
    boites = resultat.boxes.xyxy.numpy().astype(int)

    visages = []
    for x1, y1, x2, y2 in boites:
        visage = image.crop((x1, y1, x2, y2))        # 1. découpe
        visage = visage.convert("L")                 # 2. niveaux de gris
        visage = visage.resize((48, 48))             # 3. 48x48
        visage = np.array(visage) / 255.0            # 4. pixels entre 0 et 1
        visages.append(visage.reshape(48, 48, 1))    # 5. bonne forme

    if len(visages) == 0:
        return boites, np.array(visages), np.zeros((0, len(classes)))

    visages = np.array(visages)
    # Appel direct plutôt que .predict(), qui est lent sur peu d'images (gênant pour la webcam)
    probas = modele(visages, training=False).numpy()
    return boites, visages, probas

### 8.2 Résultat sur une photo

In [ ]:
boites, visages, probas = analyser_image(photo)

textes = [f"{classes[np.argmax(p)]} {p.max():.2f}" for p in probas]
dessiner_boites(photo, boites, textes, "Détection + expression")

### 8.3 Ce que voit le CNN

Pour chaque visage, on affiche la découpe que reçoit le CNN (48×48, gris) et les 7 probas du Softmax. Ça permet de vérifier que les découpes ressemblent aux images de FER-2013.

In [ ]:
fig, axes = plt.subplots(len(visages), 2, figsize=(10, 3 * len(visages)), squeeze=False)

for i in range(len(visages)):
    axes[i, 0].imshow(visages[i].squeeze(), cmap="gray")
    axes[i, 0].set_title(f"Visage {i + 1} (48×48)")
    axes[i, 0].axis("off")

    axes[i, 1].barh(classes, probas[i])
    axes[i, 1].set_xlim(0, 1)
    axes[i, 1].set_title(f"Prédiction : {classes[np.argmax(probas[i])]}")

plt.tight_layout()
plt.show()

Le visage de gauche (qui crie, sourcils froncés) est prédit angry à 0.56, ce qui colle avec l'image. Pour celui de droite, plus neutre, le CNN hésite entre sad, angry, fear et neutral (~0.2 chacun).

Les découpes ressemblent aux images de FER-2013, mais le cadrage de YOLO est un peu différent, et le visage de gauche est de profil alors que FER-2013 contient surtout des visages de face.

### 8.4 Sur vos propres photos

Pour tester une autre photo sur Colab, glissez-la dans le panneau "Fichiers" à gauche et changez le chemin ci-dessous. Par défaut, on prend la 2e photo d'exemple d'ultralytics.

In [ ]:
chemin_photo = ASSETS / "bus.jpg"   # remplacer par "ma_photo.jpg" par exemple

photo2 = Image.open(chemin_photo).convert("RGB")
boites, visages, probas = analyser_image(photo2)

print("Visages détectés :", len(boites))
textes = [f"{classes[np.argmax(p)]} {p.max():.2f}" for p in probas]
dessiner_boites(photo2, boites, textes, "Détection + expression")

## Partie 9 : Reconnaissance en direct

Une vidéo est une suite d'images, donc on applique la chaîne de la partie 8 à chaque image de la webcam. On utilise OpenCV (`cv2`) pour lire la webcam et afficher le résultat.

Webcam → images → YOLO → découpe des visages → CNN → affichage

### 9.1 Le problème de la vitesse

La webcam donne ~30 images par seconde, mais sur CPU YOLO prend ~40 ms par image et le CNN ~20 ms. En analysant toutes les images, on serait vers 15 images par seconde au mieux et la vidéo saccaderait.

On affiche donc toutes les images, mais on n'en analyse qu'une sur N. Entre deux analyses, on redessine les derniers résultats. La vidéo est plus fluide et les émotions sont quand même mises à jour plusieurs fois par seconde. Le défaut, c'est que la boîte suit le visage avec un peu de retard si on bouge vite.

### 9.2 La boucle en direct

À lancer en local (VS Code ou Jupyter). Sur Colab il n'y a pas de webcam, donc la cellule s'arrête avec un message.

Dans la fenêtre, `s` enregistre une capture et `q` quitte. On peut aussi mettre le chemin d'une vidéo à la place de `0` dans `source`.

In [1]:
import sys
import time
import cv2

source = 0               # 0 = webcam, ou "ma_video.mp4"
analyser_tous_les = 3    # on analyse 1 image sur 3

if "google.colab" in sys.modules:
    print("Pas de webcam ni de fenêtre sur Colab : lancez cette partie en local.")
else:
    lecteur = cv2.VideoCapture(source)
    numero = 0
    boites, probas = [], []
    fps = 0
    debut = time.time()

    # try/finally pour libérer la webcam même si on arrête la cellule
    try:
        while True:
            ok, frame = lecteur.read()
            if not ok:   # webcam indisponible ou fin de la vidéo
                break

            # Analyse seulement une image sur N, sinon on garde les derniers résultats
            if numero % analyser_tous_les == 0:
                # OpenCV donne les images en BGR, notre chaîne attend du RGB
                image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                boites, visages, probas = analyser_image(image)

            for (x1, y1, x2, y2), p in zip(boites, probas):
                texte = f"{classes[np.argmax(p)]} {p.max():.2f}"
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, texte, (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            numero += 1
            fps = numero / (time.time() - debut)
            cv2.putText(frame, f"{fps:.1f} images/s", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            cv2.imshow("Expressions faciales - s : capture, q : quitter", frame)
            touche = cv2.waitKey(1) & 0xFF
            if touche == ord("s"):
                cv2.imwrite("capture_webcam.png", frame)
                print("Capture enregistrée dans capture_webcam.png")
            if touche == ord("q"):
                break

    finally:
        lecteur.release()
        cv2.destroyAllWindows()
    if numero == 0:
        print("Impossible de lire la webcam : elle est peut-être utilisée par une autre appli (Teams, Zoom...)")
        print("ou bloquée dans les paramètres de confidentialité de Windows.")
    else:
        print(f"{numero} images affichées, {fps:.1f} images/s en moyenne")

NameError: name 'Image' is not defined

### 9.3 Une capture

La dernière capture enregistrée avec `s` :

In [1]:
if os.path.exists("capture_webcam.png"):
    plt.figure(figsize=(9, 6))
    plt.imshow(Image.open("capture_webcam.png").convert("RGB"))
    plt.axis("off")
    plt.show()
else:
    print("Pas encore de capture : appuyez sur 's' pendant la démo.")

NameError: name 'os' is not defined

### 9.4 Limites

- L'image analysée fait une petite pause dans l'affichage. Faire l'analyse dans un thread séparé ou sur GPU réglerait ça.
- Chaque analyse est indépendante des autres, donc si le CNN hésite, l'étiquette peut changer d'une analyse à l'autre.
- Le CNN a appris sur des visages de face bien cadrés. Avec un mauvais éclairage, un visage de profil ou loin de la caméra, les prédictions sont moins fiables.

## Conclusion

On a construit un système complet de reconnaissance d'expressions : YOLO trouve les visages sur une image ou la webcam, on les prétraite, puis notre CNN prédit l'expression.

| Modèle | Accuracy test | Loss test |
|---|---|---|
| Toujours répondre "happy" | ~0.25 | |
| Réseau dense (partie 2) | 0.380 | 1.61 |
| CNN de base (partie 3) | 0.521 | 3.19 |
| CNN final avec Dropout (partie 6) | 0.576 | 1.13 |

Le plus gros gain vient du passage au CNN (+14 points), parce que les convolutions repèrent les yeux, la bouche et les sourcils où qu'ils soient. Le Dropout corrige ensuite le surapprentissage (+5.5 points, et une loss presque divisée par 3). Le kernel 5×5 et l'arrêt plus tôt n'ont pas fait mieux.

Le modèle reconnaît bien happy et surprise, mais confond encore fear, sad et neutral, et a du mal avec disgust qui a très peu d'images. Une partie des erreurs vient aussi des labels du dataset. En direct, le système tourne sur un simple CPU mais il est moins fiable sur les visages de profil ou mal éclairés.

Pour aller plus loin, on pourrait pondérer la loss avec `class_weight` pour aider les petites classes, entraîner le modèle final un peu plus longtemps, ou ajouter de la data augmentation.